# BARRED — sentiment analysis demo

Generate a labeled sentiment dataset from a single criterion and a few unlabeled examples.

## Installation

`barred` builds on [any-llm](https://docs.mozilla.ai/any-llm/), which ships **no provider SDK by
default**.

You need the `any-llm-sdk` extra for the provider you intend to use.

For example:

| Provider      | Install                                       |
|---------------|-----------------------------------------------|
| OpenAI        | `pip install barred "any-llm-sdk[openai]"`    |
| Anthropic     | `pip install barred "any-llm-sdk[anthropic]"` |
| Gemini        | `pip install barred "any-llm-sdk[gemini]"`    |
| All providers | `pip install barred "any-llm-sdk[all]"`       |

Check the [list of providers](https://docs.mozilla.ai/providers).

In [ ]:
import contextlib
import os
import sys

if "google.colab" in sys.modules:
    # any-llm ships no provider: trim or extend this list to match your key
    %pip install -q "git+https://github.com/tomsquest/barred" \
        "any-llm-sdk[openai,anthropic,gemini,vertexai]" genai-prices

    from google.colab import userdata  # ty: ignore[unresolved-import]

    # Secrets panel (key icon, left sidebar) -> environment, where any-llm looks
    for name in (
        "OPENAI_API_KEY",
        "ANTHROPIC_API_KEY",
        "GEMINI_API_KEY",
        "GCP_PROJECT",
        "GCP_LOCATION",
    ):
        with contextlib.suppress(Exception):
            os.environ[name] = userdata.get(name)
else:
    from dotenv import load_dotenv

    load_dotenv()

## Pick your provider

Set one of `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, or `GCP_PROJECT` in your environment. The first one found wins — reorder the branches to force another.

In [ ]:
from typing import Any

from any_llm import LLMProvider

from barred import LLM

OPENAI_MODEL = "gpt-5.6-terra"
ANTHROPIC_MODEL = "claude-sonnet-5"
GEMINI_MODEL = "gemini-3.7-flash"
VERTEX_MODEL = "gemini-3.7-flash"

kwargs: dict[str, Any] = {}
if os.getenv("OPENAI_API_KEY"):
    provider, model = LLMProvider.OPENAI, OPENAI_MODEL
elif os.getenv("ANTHROPIC_API_KEY"):
    provider, model = LLMProvider.ANTHROPIC, ANTHROPIC_MODEL
elif os.getenv("GEMINI_API_KEY"):
    provider, model = LLMProvider.GEMINI, GEMINI_MODEL
elif os.getenv("GCP_PROJECT"):
    provider, model = LLMProvider.VERTEXAI, VERTEX_MODEL
    kwargs = {
        "project": os.environ["GCP_PROJECT"],
        "location": os.getenv("GCP_LOCATION", "global"),
    }
else:
    msg = "Set one of OPENAI_API_KEY, ANTHROPIC_API_KEY, GEMINI_API_KEY, or GCP_PROJECT"
    raise RuntimeError(msg)

llm = LLM(provider=provider, model=model, **kwargs)
print(f"Using {provider.value} / {model}")

In [ ]:
from genai_prices import Usage, calc_price


def print_usage() -> None:
    """Tokens and price accumulated since the LLM was created."""
    usage = llm.total_usage
    price = calc_price(
        Usage(
            input_tokens=usage.input_tokens,
            output_tokens=usage.output_tokens,
            cache_read_tokens=usage.cached_tokens,
        ),
        provider_id=provider.value,
        model_ref=model,
    )
    print(
        f"Usage: {usage.total_tokens} tokens ({usage.input_tokens} in, {usage.output_tokens} out, {usage.cached_tokens} cached)."
    )
    print(f"Estimated price: ${price.total_price:.4f}")

## Step 1. Task definition

In [ ]:
from barred import Criterion, Example

criterion: Criterion = (
    "True when the sentence expresses a positive sentiment, False otherwise"
)

# Unlabeled: they anchor the domain and style, never the label
examples: list[Example] = [
    "The delivery arrived two days late and the box was crushed.",
    "Honestly one of the best purchases I've made this year.",
    "It works, I guess.",
]

## Step 2. Decompose the Criterion into Dimensions

In [ ]:
from barred import decompose_dimensions

dimensions = await decompose_dimensions(llm, criterion=criterion, examples=examples)

for decomposed in dimensions:
    print(f"\n{decomposed.dimension.name}: {decomposed.dimension.description}")
    for instantiation in decomposed.instantiations:
        print(f"  - {instantiation.description}")

print_usage()

## Step 3. Generate Samples

In [ ]:
from barred import barred

samples = [
    sample
    async for sample in barred(
        llm,
        criterion=criterion,
        examples=examples,
        dimensions=dimensions,
        num_samples=5,
    )
]

for sample in samples:
    print(f"{sample.label}\t{sample.input_block}")

print_usage()

## Bonus — Observe the events

Pass an observer to the `LLM` and you see everything that happens: decomposition, draws, debates, refinements, LLM calls.

By default, a silent observer is used (NullObserver).
A logging observer is shipped with Barred, see below.
You can implement your own by inheriting from NullObserver.

In [ ]:
import logging
import sys

from barred import LLM, LoggingObserver

# LoggingObserver writes to the "barred" logger, silent until it is given a handler.
# INFO = the milestones, DEBUG = the prompts and the responses.
logging.getLogger("barred").handlers = [logging.StreamHandler(sys.stdout)]
logging.getLogger("barred").setLevel(logging.INFO)
logging.getLogger("barred").propagate = False

llm = LLM(
    provider=provider,
    model=model,
    observer=LoggingObserver(),  # <-- the observer
    **kwargs,
)

# Now use the lib
# decompose_dimensions(llm, ...)
# barred(llm, ...)